### OpenAI Agents SDK

This notebook walks through a compact agent workflow that uses structured outputs and an input guardrail to keep the interaction focused and safe.

Helpful references:
- SDK docs: https://openai.github.io/openai-agents-python/
- Guardrails docs: https://openai.github.io/openai-agents-python/guardrails/
- Structured Output: https://openai.github.io/openai-agents-python/ref/agent_output/
- GitHub repo: https://github.com/openai/openai-agents-python

- Pydantic Docs: https://pydantic.dev/docs/validation/latest/concepts/models/
- Base Model Docs: https://pydantic.dev/docs/validation/latest/api/pydantic/base_model/#pydantic.BaseModel
- Pydantic Code: https://github.com/pydantic/pydantic/blob/main/pydantic/main.py?utm_source=chatgpt.com


In [ ]:
!pip install -q openai-agents python-dotenv

In [48]:
from dotenv import load_dotenv
from agents import (
  Agent, Runner, GuardrailFunctionOutput, input_guardrail)
from IPython.display import Markdown, display
from pydantic import BaseModel
import os


##### Packages Used

- `openai-agents` for `Agent`, `Runner`, `trace`, `GuardrailFunctionOutput`, `input_guardrail`
- `python-dotenv` for `load_dotenv`
- `pydantic` for `BaseModel`

In [50]:
load_dotenv()

print("OpenAI API key loaded:", os.getenv("OPENAI_API_KEY") is not None)

OpenAI API key loaded: True


Use this accounts to get started.
- https://platform.openai.com/login

### Guardrails

This guardrail checks whether the request is food-related before the main agent runs. A small structured output makes the decision easy to read and reason about.


In [57]:
class FoodCheckOutput(BaseModel):
        is_food_in_message: bool
        food: str

guardrail_agent = Agent(
        name="Guardrail Agent",
        instructions="Check if the user is asking about food.",
        model="gpt-4o-mini",
        output_type=FoodCheckOutput
)

In [59]:
result = await Runner.run(guardrail_agent, "tell me about apples")
print(result.final_output)

is_food_in_message=True food='apples'


In [60]:
@input_guardrail
async def guardrail_food_check(ctx, agent, message):
  result = await Runner.run(guardrail_agent, message, context=ctx.context)
  is_food_in_message = result.final_output.is_food_in_message
  return GuardrailFunctionOutput(output_info={"found_food": result.final_output},tripwire_triggered=is_food_in_message)

# Structured Output

Pydantic defines the expected output shape for the agent, and `BaseModel` enforces it —
ensuring every result follows the same structure, with the right fields and types.
This keeps results consistent and easy to work with in code.

Inheriting from `BaseModel` gives you validation, JSON conversion, and schema generation —
a plain class with type hints alone does not.


In [51]:
class Finding(BaseModel):
    subtopic: str
    details: str

class ResearchOutput(BaseModel):
    topic: str
    findings: list[Finding]
    


### Create Agents

Define the researcher and reporter agents. The researcher will use both the structured output schema and the guardrail.


In [52]:
researcher_inst = "You are a skilled and resourceful researcher. Your job is to deeply investigate any assigned topic"

This helper turns the Pydantic model into a nice summary block.

In [61]:
researcher = Agent(
        name="Professional Researcher",
        instructions=researcher_inst,
        model="gpt-4o-mini",
        input_guardrails=[guardrail_food_check],
        output_type=ResearchOutput,
)

In [64]:
topic = "atlantic ocean"

In [65]:
result = await Runner.run(
    researcher, 
    f"Research the topic: {topic}. Output 3-4 detailed bullets about your findings."
)
research_result = result.final_output


In [56]:
print(research_result)

topic='Atlantic Ocean' findings=[Finding(subtopic='Geography', details="The Atlantic Ocean is the second largest ocean, covering about 20% of the Earth's surface and spanning approximately 106.46 million square kilometers. It separates North and South America from Europe and Africa, and is connected to the Arctic Ocean in the north and the Southern Ocean in the south."), Finding(subtopic='Biodiversity', details='The Atlantic Ocean hosts a diverse range of marine life, including species such as the North Atlantic right whale, various sharks (like the great white and hammerhead), and numerous fish species. Coral reefs, particularly in the Caribbean Sea, are crucial ecosystems that support over 4,000 fish species.'), Finding(subtopic='Economic Importance', details='The Atlantic Ocean is a vital resource for global trade, with major shipping routes traversing its waters. It is also rich in natural resources, including oil and gas reserves, particularly along the continental shelf, and is a

In [46]:
for finding in research_result.findings:
    print(f"Subtopic: {finding.subtopic}, Details: {finding.details}")

Subtopic: Geography, Details: The Atlantic Ocean is the second-largest ocean in the world, covering approximately 41 million square miles (106 million square kilometers). It is bordered by North America to the west, Europe and Africa to the east, and South America to the south. Its major features include the Mid-Atlantic Ridge, which is the longest mountain range in the world, and the Bermuda Triangle, an area infamous for the mysterious disappearances of ships and aircraft.
Subtopic: Ecosystem, Details: The Atlantic Ocean hosts a diverse range of ecosystems, including coral reefs, mangroves, and deep-sea environments. It is home to a variety of marine life, including commercially important species such as cod, herring, and tuna, as well as iconic species like whales, dolphins, and sea turtles. The biodiversity of the Atlantic is vital for both ecological balance and human economic activities such as fishing and tourism.
Subtopic: Climate Influence, Details: The Atlantic Ocean plays a 

### Format

Convert the structured findings into a readable report for the final display.


In [67]:
def format_for_summary(result: ResearchOutput) -> str:
    sections = "\n\n".join(
        f"## {f.subtopic}\n{f.details}" for f in result.findings
    )
    return f"# {result.topic}\n\n{sections}"

In [68]:
formatted_research = format_for_summary(research_result)
print(formatted_research)

# Atlantic Ocean

## Geography and Size
The Atlantic Ocean is the second-largest ocean, covering approximately 41 million square miles (106 million square kilometers) and spanning from the Arctic Ocean in the north to the Southern Ocean in the south. It borders multiple continents, including North America, South America, Europe, and Africa, making it crucial for global trade routes.

## Biodiversity
The Atlantic Ocean hosts a rich variety of marine life, including species such as whales, dolphins, and numerous fish varieties. The ocean's different zones, such as the deep-sea environments and coastal areas, contribute to highly diverse ecosystems. Coral reefs, particularly in the Caribbean, are among the most productive ecosystems within the Atlantic.

## Current Systems and Climate Influence
The Atlantic Ocean plays a fundamental role in Earth's climate through the Atlantic Meridional Overturning Circulation (AMOC), which regulates temperatures and weather patterns, especially in the N

In [69]:
print(research_result.model_dump_json(indent=2))

{
  "topic": "Atlantic Ocean",
  "findings": [
    {
      "subtopic": "Geography and Size",
      "details": "The Atlantic Ocean is the second-largest ocean, covering approximately 41 million square miles (106 million square kilometers) and spanning from the Arctic Ocean in the north to the Southern Ocean in the south. It borders multiple continents, including North America, South America, Europe, and Africa, making it crucial for global trade routes."
    },
    {
      "subtopic": "Biodiversity",
      "details": "The Atlantic Ocean hosts a rich variety of marine life, including species such as whales, dolphins, and numerous fish varieties. The ocean's different zones, such as the deep-sea environments and coastal areas, contribute to highly diverse ecosystems. Coral reefs, particularly in the Caribbean, are among the most productive ecosystems within the Atlantic."
    },
    {
      "subtopic": "Current Systems and Climate Influence",
      "details": "The Atlantic Ocean plays a f

#### Review the Traces

https://platform.openai.com/logs?api=traces
